# Wave manipulation explorer

Select a TIMIT WAV file, look at its waveform and its short-time FFT (spectrogram), and listen to both.

The short-time FFT is computed as the models' front-end does it: `compute_linear_spectrogram` from `src/data/features.py`
(frames of `frame_length_s`, every `frame_step_s`, windowed, FFT of `nfft` points, magnitude), then `apply_drc`,
all with the `TransformationConfig` defaults.

The short-time FFT keeps only each bin's magnitude, the phase is dropped (as for the models). To listen to it, Griffin-Lim
estimates a phase that fits the magnitudes and turns the result back into a waveform.

**Voice activity detection (VAD)** marks the frames that contain speech: a frame (the same frames as the short-time FFT)
is speech if its energy is at most `threshold` dB below the loudest frame's. Pauses shorter than `min pause` are counted
as speech, so stop closures and short breaks inside words aren't cut out. The player plays only the speech, with the
non-speech removed.

In [1]:
import base64
import io
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from torchaudio.transforms import GriffinLim

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import TransformationConfig
from src.data.features import apply_drc, compute_linear_spectrogram

In [2]:
timit_root = Path("~/.cache/datasets/timit").expanduser()
# Labelled relative to the TIMIT/ folder, e.g. "TRAIN/DR1/FCJF0/SA1.WAV".
wav_paths = [(str(p.relative_to(p.parents[3])), p) for p in sorted(timit_root.rglob("*")) if p.suffix.lower() == ".wav"]
len(wav_paths)

6300

In [3]:
config = TransformationConfig()
wav_path = widgets.Dropdown(options=wav_paths, description="WAV file", layout=widgets.Layout(width="500px"))

# The figures (as PNGs) and the audio players go into HTML widgets: plot and Audio outputs
# don't render inside an Output widget in PyCharm.
wave_plot = widgets.HTML()
wave_player = widgets.HTML()
stft_plot = widgets.HTML()
stft_player = widgets.HTML()
threshold = widgets.FloatSlider(value=-30, min=-60, max=0, step=1, description="threshold [dB]")
min_pause = widgets.IntSlider(value=100, min=0, max=500, step=10, description="min pause [ms]")
vad_plot = widgets.HTML()
vad_player = widgets.HTML()


def new_figure():
    """A figure with a plot and a narrow column for a colour bar. All figures use the same size and margins,
    so their plots get the same width and their time axes line up."""
    fig, (ax, ax_colorbar) = plt.subplots(1, 2, figsize=(12, 4), width_ratios=(1, 0.015))
    fig.subplots_adjust(left=0.07, right=0.93, bottom=0.13, top=0.92, wspace=0.02)
    return fig, ax, ax_colorbar


def to_html(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=100)
    plt.close(fig)
    return f'<img src="data:image/png;base64,{base64.b64encode(buffer.getvalue()).decode()}" style="max-width: 100%">'


def show(wav_path):
    waveform, sample_rate = sf.read(str(wav_path), dtype="float32")
    assert sample_rate == config.sample_rate, f"{wav_path} is {sample_rate} Hz, config expects {config.sample_rate} Hz"
    duration = len(waveform) / sample_rate

    fig, ax, ax_colorbar = new_figure()
    ax_colorbar.axis("off")
    ax.plot(np.arange(len(waveform)) / sample_rate, waveform, linewidth=0.5)
    ax.set_title(f"waveform: {wav_path.relative_to(wav_path.parents[3])} ({len(waveform)} samples, {sample_rate} Hz)")
    ax.set_xlabel("time [s]")
    ax.set_ylabel("amplitude")
    ax.set_xlim(0, duration)
    ax.grid(alpha=0.3)
    wave_plot.value = to_html(fig)
    wave_player.value = Audio(waveform, rate=sample_rate)._repr_html_()

    # (frames, nfft // 2 + 1): the FFT magnitude of every frame.
    magnitude = compute_linear_spectrogram(waveform, config)
    stft = apply_drc(magnitude)  # compressed to a log-like scale for plotting

    fig, ax, ax_colorbar = new_figure()
    # Frame i covers [i * frame_step_s, i * frame_step_s + frame_length_s]; its column is drawn at the frame's centre.
    # Bin k is k * sample_rate / nfft Hz.
    first_centre = config.frame_length_s / 2
    image = ax.imshow(stft.T, origin="lower", aspect="auto", cmap="magma",
                      extent=(first_centre - config.frame_step_s / 2,
                              first_centre + (stft.shape[0] - 0.5) * config.frame_step_s, 0, sample_rate / 2))
    ax.set_title(f"short-time FFT: {stft.shape[0]} frames x {stft.shape[1]} bins "
                 f"({config.frame_length_s * 1000:g} ms {config.window} window every {config.frame_step_s * 1000:g} ms, nfft {config.nfft})")
    ax.set_xlabel("time [s]")
    ax.set_ylabel("frequency [Hz]")
    ax.set_xlim(0, duration)
    fig.colorbar(image, cax=ax_colorbar, label="DRC level")
    stft_plot.value = to_html(fig)

    # Griffin-Lim with the same frames and window as compute_linear_spectrogram (power=1: magnitudes, not powers).
    # It assumes frame i is centred on sample i * frame_step, but frame i starts there, so its output is shifted
    # forward by half a frame: padding that many zeros in front puts it back in place.
    griffin_lim = GriffinLim(n_fft=config.nfft, win_length=config.frame_length, hop_length=config.frame_step,
                             window_fn=torch.hamming_window if config.window == "hamming" else torch.hann_window,
                             power=1.0)
    torch.manual_seed(0)  # Griffin-Lim starts from random phases: the same file always sounds the same.
    reconstruction = griffin_lim(torch.from_numpy(magnitude.T)).numpy()
    reconstruction = np.pad(reconstruction, (config.frame_length // 2, 0))
    stft_player.value = Audio(reconstruction, rate=sample_rate)._repr_html_()


def runs(mask):
    """(start, end, value) of every run of equal values in the boolean array `mask`, end exclusive."""
    starts = np.r_[0, np.flatnonzero(np.diff(mask)) + 1]
    ends = np.r_[starts[1:], len(mask)]
    return [(start, end, mask[start]) for start, end in zip(starts, ends)]


def show_vad(wav_path, threshold, min_pause):
    waveform, sample_rate = sf.read(str(wav_path), dtype="float32")
    duration = len(waveform) / sample_rate

    # Energy of every frame (the frames compute_linear_spectrogram cuts), in dB relative to the loudest frame.
    frames = np.lib.stride_tricks.sliding_window_view(waveform, config.frame_length)[::config.frame_step]
    energy = 10 * np.log10(np.mean(frames ** 2, axis=1) + 1e-10)
    energy -= energy.max()
    speech = energy >= threshold
    # Bridge pauses shorter than min_pause between two speech runs.
    for start, end, is_speech in runs(speech):
        if not is_speech and 0 < start and end < len(speech) and (end - start) * config.frame_step_s * 1000 < min_pause:
            speech[start:end] = True

    # Frames overlap, so each frame stands for the frame_step samples around its centre.
    offset = (config.frame_length - config.frame_step) // 2
    speech_runs = [(start * config.frame_step + offset, end * config.frame_step + offset)
                   for start, end, is_speech in runs(speech) if is_speech]
    kept = np.concatenate([waveform[start:end] for start, end in speech_runs])

    fig, ax, ax_colorbar = new_figure()
    ax_colorbar.axis("off")
    centres = config.frame_length_s / 2 + np.arange(len(energy)) * config.frame_step_s
    ax.plot(centres, energy, linewidth=1, label="frame energy")
    ax.axhline(threshold, color="tab:red", linestyle="--", linewidth=1, label="threshold")
    for i, (start, end) in enumerate(speech_runs):
        ax.axvspan(start / sample_rate, end / sample_rate, color="tab:green", alpha=0.2, label="speech" if i == 0 else None)
    ax.set_title(f"VAD: {len(speech_runs)} speech segments, {len(kept) / sample_rate:.2f} of {duration:.2f} s kept")
    ax.set_xlabel("time [s]")
    ax.set_ylabel("energy [dB re loudest frame]")
    ax.set_xlim(0, duration)
    ax.set_ylim(max(energy.min(), -80) - 3, 3)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right")
    vad_plot.value = to_html(fig)
    vad_player.value = Audio(kept, rate=sample_rate)._repr_html_()


# Two outputs, so moving a VAD slider doesn't redo the short-time FFT and Griffin-Lim.
output = widgets.interactive_output(show, dict(wav_path=wav_path))
vad_output = widgets.interactive_output(show_vad, dict(wav_path=wav_path, threshold=threshold, min_pause=min_pause))
display(widgets.VBox([wav_path, wave_plot, wave_player, stft_plot, stft_player,
                      widgets.HBox([threshold, min_pause]), vad_plot, vad_player, output, vad_output]))